04_sentiment_analysis_gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/04_sentiment_analysis_gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/04_sentiment_analysis_gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Sentiment Analysis with Gemini: Customer Feedback at Scale

## What you will learn
- Use an LLM for sentiment analysis on real review data.
- Compare binary (positive/negative) vs. nuanced (5-point scale) sentiment.
- See how prompt framing changes sentiment assignments.

**NLP Task**: Sentiment & Opinion Analysis — gauging attitudes and opinions in text.

**Dataset**: `rotten_tomatoes` from HuggingFace — movie reviews labelled positive/negative.
We reframe it as a product review analysis scenario.

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai datasets pandas


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Load Dataset


In [ ]:
from datasets import load_dataset

ds = load_dataset('rotten_tomatoes', split='test')
LABEL_MAP = {0: 'negative', 1: 'positive'}

import random
random.seed(42)

# Get 10 positive, 10 negative
pos_idx = [i for i in range(len(ds)) if ds[i]['label'] == 1]
neg_idx = [i for i in range(len(ds)) if ds[i]['label'] == 0]
sample_indices = random.sample(pos_idx, 10) + random.sample(neg_idx, 10)
random.shuffle(sample_indices)

sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df['ground_truth'] = sample_df['label'].map(LABEL_MAP)

print(f'Sample: {len(sample_df)} reviews')
print(f'Distribution: {dict(sample_df["ground_truth"].value_counts())}')


## Approach 1: Binary Sentiment Classification

Simple positive/negative classification — the most common business use case.


In [ ]:
BINARY_PROMPT = """Classify the sentiment of this product review as either "positive" or "negative".
Respond with ONLY the word "positive" or "negative".

Review: "{text}"

Sentiment:"""

def classify_binary(text: str) -> str:
    prompt = BINARY_PROMPT.format(text=text)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        result = resp.text.strip().lower()
        if 'positive' in result:
            return 'positive'
        elif 'negative' in result:
            return 'negative'
        return result
    except Exception as e:
        return f'error: {e}'

binary_results = []
for _, row in sample_df.iterrows():
    binary_results.append(classify_binary(row['text']))
    time.sleep(0.3)

sample_df['binary_pred'] = binary_results
sample_df['binary_correct'] = sample_df['binary_pred'] == sample_df['ground_truth']

print(f'Binary Accuracy: {sample_df["binary_correct"].mean():.0%}')


## Approach 2: Nuanced 5-Point Scale

Same reviews, but now we ask for a 1-5 rating.
This simulates: "What's the overall sentiment trend across this month's reviews?"


In [ ]:
SCALE_PROMPT = """Rate the sentiment of this review on a scale of 1-5:
1 = Very Negative, 2 = Negative, 3 = Neutral, 4 = Positive, 5 = Very Positive

Review: "{text}"

Respond with ONLY a single number (1, 2, 3, 4, or 5):"""

def classify_scale(text: str) -> int:
    prompt = SCALE_PROMPT.format(text=text)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        result = resp.text.strip()
        # Extract first digit
        for char in result:
            if char.isdigit() and char in '12345':
                return int(char)
        return -1
    except Exception:
        return -1

scale_results = []
for _, row in sample_df.iterrows():
    scale_results.append(classify_scale(row['text']))
    time.sleep(0.3)

sample_df['scale_pred'] = scale_results
# Map scale to binary for comparison: 1-2 = negative, 4-5 = positive, 3 = neutral
sample_df['scale_as_binary'] = sample_df['scale_pred'].map(
    lambda x: 'negative' if x <= 2 else ('positive' if x >= 4 else 'neutral')
)
sample_df['scale_agrees_gt'] = sample_df['scale_as_binary'] == sample_df['ground_truth']

print(f'Scale predictions: {dict(sample_df["scale_pred"].value_counts().sort_index())}')
print(f'Scale→Binary agreement with ground truth: {sample_df["scale_agrees_gt"].mean():.0%}')


## Compare Approaches Side-by-Side


In [ ]:
print(f'{"Review (first 60 chars)":<65} {"GT":<10} {"Binary":<10} {"Scale":<6} {"Match?"}')
print('-' * 100)
for _, row in sample_df.iterrows():
    text_short = row['text'][:60] + '...'
    match = '✓' if row['binary_correct'] else '✗'
    print(f'{text_short:<65} {row["ground_truth"]:<10} {row["binary_pred"]:<10} {row["scale_pred"]:<6} {match}')


## Approach 3: The "Lottery of Prompting" — Same Task, Different Prompt

Let's rephrase the same classification task to see if results change.
This directly demonstrates prompt sensitivity from the lecture.


In [ ]:
ALT_PROMPT = """You are analyzing customer feedback for a product team.
Does this review express satisfaction or dissatisfaction with the product?
Answer with ONLY "satisfied" or "dissatisfied".

Feedback: "{text}"

Assessment:"""

def classify_alt(text: str) -> str:
    prompt = ALT_PROMPT.format(text=text)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        result = resp.text.strip().lower()
        if 'satisf' in result and 'dissatisf' not in result:
            return 'positive'
        elif 'dissatisf' in result:
            return 'negative'
        return result
    except Exception:
        return 'error'

alt_results = []
for _, row in sample_df.iterrows():
    alt_results.append(classify_alt(row['text']))
    time.sleep(0.3)

sample_df['alt_pred'] = alt_results
sample_df['alt_correct'] = sample_df['alt_pred'] == sample_df['ground_truth']

print(f'\n=== Prompt Comparison ===')
print(f'Binary prompt accuracy:      {sample_df["binary_correct"].mean():.0%}')
print(f'Alternative prompt accuracy:  {sample_df["alt_correct"].mean():.0%}')

# Where do they disagree?
disagreements = sample_df[sample_df['binary_pred'] != sample_df['alt_pred']]
print(f'\nPrompts disagree on {len(disagreements)} reviews:')
for _, row in disagreements.iterrows():
    print(f'  "{row["text"][:70]}..."')
    print(f'    Binary: {row["binary_pred"]}  |  Alt: {row["alt_pred"]}  |  Truth: {row["ground_truth"]}')


## Discussion Questions

1. **Does the assigned sentiment match what a human reader would conclude?**
   Look at the errors — are they truly wrong, or is the ground truth debatable?

2. **How did prompt framing affect results?**
   "positive/negative" vs "satisfied/dissatisfied" — same task, different words.

3. **Binary vs. scale: which is better for business?**
   Binary is easier to evaluate. Scale gives more nuance but more room for error.

## Export


In [ ]:
export_df = sample_df[['text', 'ground_truth', 'binary_pred', 'binary_correct',
                        'scale_pred', 'alt_pred', 'alt_correct']].copy()
export_df['student_notes'] = ''
export_path = 'sentiment_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
